In [1]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_voyageai import VoyageAIEmbeddings
from langchain_chroma import Chroma

C:\Users\jains\AppData\Local\Temp\ipykernel_37860\2573709327.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [6]:
from dotenv import load_dotenv
load_dotenv()

False

In [7]:
import pandas as pd
import numpy as np

In [5]:
reviews_3UP = pd.read_parquet("data/reviews_3UP.parquet")
restaurant_docs = pd.read_parquet("data/restaurant_docs.parquet")
reviews  = pd.read_parquet("data/reviews_clean.parquet")
business = pd.read_parquet("data/business_clean.parquet")
mapped   = pd.read_parquet("data/mapped.parquet")
user     = pd.read_parquet("data/user_clean.parquet")
tip      = pd.read_parquet("data/tip_clean.parquet")
checkin  = pd.read_parquet("data/checkin_clean.parquet")

In [8]:
biz_cols = business[["business_id", "name", "city", "state", "postal_code","latitude", "longitude", "stars", "review_count","attributes", "categories"]].rename(columns={"stars": "biz_stars","review_count": "biz_review_count"})

user_cols = user[["user_id", "review_count", "average_stars", "yelping_since"]].rename(columns={"review_count": "user_review_count"})

df = (reviews_3UP.merge(biz_cols,  on="business_id", how="left").merge(user_cols, on="user_id", how="left").merge(checkin[["business_id", "checkinCount", "latestCheckin"]], on="business_id", how="left"))

In [9]:
import ast

def parse_attrs(a):
    if not isinstance(a, dict): return {}
    out = {}
    for k, v in a.items():
        try: out[k] = ast.literal_eval(v) if isinstance(v, str) and v[:1] in "{u'TF" else v
        except Exception: out[k] = v
    return out

attrs = df["attributes"].map(parse_attrs)
df["price_range"]  = attrs.map(lambda d: d.get("RestaurantsPriceRange2"))   # 1–4 ($ to $$$$)
df["takeout"]      = attrs.map(lambda d: d.get("RestaurantsTakeOut") == True)
df["delivery"]     = attrs.map(lambda d: d.get("RestaurantsDelivery") == True)
df["reservations"] = attrs.map(lambda d: d.get("RestaurantsReservations") == True)
df["outdoor"]      = attrs.map(lambda d: d.get("OutdoorSeating") == True)
df["good_for_kids"]= attrs.map(lambda d: d.get("GoodForKids") == True)

In [10]:
df

,review_id,user_id,business_id,stars,useful,text,date,wordsInDesc,lang,fsqid,...,average_stars,yelping_since,checkinCount,latestCheckin,price_range,takeout,delivery,reservations,outdoor,good_for_kids
0,KU_O5udG6zpxOg-VcAEodg,mh_-eMZ6K5RLWhZyISBhwA,XQfwVwDr-v0ZS3_CbbE5Xw,3,0,"If you decide to eat here, just be aware it is...",2018-07-07 22:09:11,101,en,13028.0,...,4.06,2016-01-13 17:20:44,177.0,2022-01-16 13:50:28,2,True,True,False,True,True
1,saUsX_uimxRlCVr67Z4Jig,8g_iMtfSiwikVnbP2etR0A,YjUWPpI6HXG530lwP-fb2A,3,0,Family diner. Had the buffet. Eclectic assortm...,2014-02-05 20:30:30,55,en,13028.0,...,4.69,2012-09-04 23:57:25,56.0,2021-11-04 17:43:20,1,True,True,True,False,True
2,AqPFMleE6RsU23_auESxiA,_7bHUi9Uuf5__HHc_Q8guQ,kxX2SOes4o-D3ZQBkiMRfA,5,1,"Wow! Yummy, different, delicious. Our favo...",2015-01-04 00:01:03,40,en,13199.0,...,4.78,2014-01-17 19:20:57,204.0,2021-05-07 21:40:57,2,True,False,True,False,True
3,Sx8TMOWLNuJBWer-0pcmoA,bcjbaE6dDog4jkNY91ncLQ,e4Vwtrqf-wpJfwesgvdgxQ,4,1,Cute interior and owner (?) gave us tour of up...,2017-01-14 20:54:15,94,en,13003.0,...,2.97,2008-04-17 13:39:54,55.0,2018-05-30 20:48:22,2,True,False,False,True,True
4,JrIxlS1TzJ-iCu79ul40cQ,eUta8W_HdHMXPzLBBZhL1A,04UD14gamNjLY0IDYVhHJg,1,1,I am a long term frequent customer of this est...,2015-09-23 23:10:31,65,en,13302.0,...,2.00,2014-06-15 18:44:27,290.0,2019-10-18 21:43:59,2,True,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
350434,lJqnUK9gASgdyx9USoKzfw,7N8IMmMrevDQ0B_Hqa87pQ,iEC0VzHM2hQ13wM7c0QwgQ,4,0,Loved the Monte Cristo sandwich. Previously ga...,2021-06-29 13:48:52,66,en,13068.0,...,2.50,2013-12-28 20:06:40,39.0,2021-10-23 20:31:46,2,True,True,False,False,False
350435,W8I3wWhb9hmglasrR6dePg,MZlZfs0ro3Aj3-FPOBPE-Q,DMmdMkh7sIhUHgVawGxY6g,5,0,This is the best BBQ in the area. The owners a...,2021-02-02 22:05:03,66,en,13068.0,...,4.17,2020-12-29 12:30:11,1.0,2021-09-11 18:06:26,NaN,True,False,False,False,False
350436,ZuMNAPcArFtaGufe-nwGOA,MVqzYt-E7y1Mdw9F_7nLjw,3XirYkP9PJvVXIEDPNNXLA,4,0,This place is hyped as one of the best places ...,2021-07-03 04:51:07,54,en,13068.0,...,4.04,2015-03-30 05:34:57,1170.0,2021-12-29 16:29:44,1,True,False,False,False,True
350437,Lohri9uZyvoNnH5z_NT6DA,vn777Y2vCynYYUYJjNYdYg,gfPDLZimZu1NtBIDbeXetg,2,1,Saw the reviews so thought I'd try this place....,2019-04-16 23:28:41,145,en,13199.0,...,4.27,2011-07-17 23:26:04,135.0,2022-01-04 23:50:40,2,True,True,True,True,True
